In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time_series.data_handlers import TimeSeriesData
from time_series.kernels import GaussianKernel
from time import time
import cvxpy as cp
import scipy as sp
from time import time
from time_series.models import RascuttiModel as RascuttiOld

2026-03-06 14:12:52.323 | INFO     | time_series.config:<module>:13 - PROJ_ROOT path is: /home/james/Repo/PhD Repo/time_series_clustering


In [2]:
def dynamics_sincos(
        theta: float,
        n_correlated_dimensions: int,
        n_uncorrelated_dimensions: int,
    ):

    n_dim = n_correlated_dimensions + n_uncorrelated_dimensions

    def f(x):
        # print(x.shape)
        x_next = np.zeros_like(x)

        for i in range(n_correlated_dimensions):
            x_next[i] += np.cos(theta * x[i]) 
            if i > 0:
                x_next[i] += -np.sin(theta*x[i-1])

            if i + 1 < n_dim:
                x_next[i] += np.sin(theta*x[i+1])

        # ------------------------------
        # Uncorrelated dimensions
        # ------------------------------
        for i in range(n_correlated_dimensions, n_dim):
            x_next[i] = np.cos(theta * x[i])

        return x_next
    return f

def time_series_generator(
    dynamics,
    x0: np.array,
    n_points: int,
    noise: float = 0.0,
):
    if n_points <= 0:
        raise ValueError("n_points must be positive")

    rng = np.random.default_rng()
    n_dim = len(x0)

    X = np.empty((n_points + 1, n_dim))
    X[0] = x0

    for t in range(n_points):
        x = X[t]
        x_next = dynamics(x)

        x_next += rng.normal(0.0, noise, size=n_dim)

        X[t + 1] = x_next

    return X


def create_dataset(
    theta: float,
    n_points: int,
    n_correlated_dimensions: int,
    n_uncorrelated_dimensions: int,
    noise: float = 0.0,
    seed: int | None = None,
):
    """
    Generate a bounded nonlinear dynamical system dataset.

    """

    if n_points <= 0:
        raise ValueError("n_points must be positive")

    rng = np.random.default_rng()

    n_dim = n_correlated_dimensions + n_uncorrelated_dimensions
    X = np.empty((n_points + 1, n_dim))
    X[0] = rng.uniform(-1.0, 1.0, size=n_dim)

    for t in range(n_points):
        x = X[t]
        x_next = np.zeros_like(x)

        # ------------------------------
        # Correlated nonlinear dynamics
        # ------------------------------
        for i in range(n_correlated_dimensions):
            x_next[i] += np.cos(theta * x[i]) 
            if i > 0:
                x_next[i] += -np.sin(theta*x[i-1])

            if i + 1 < n_dim:
                x_next[i] += np.sin(theta*x[i+1])

        # ------------------------------
        # Uncorrelated dimensions
        # ------------------------------
        for i in range(n_correlated_dimensions, n_dim):
            x_next[i] = np.cos(theta * x[i])

        # ------------------------------
        # Damping + noise (bounded step)
        # ------------------------------
        x_next += rng.normal(0.0, noise, size=n_dim)

        X[t + 1] = x_next

    return X


In [3]:
dynamics = dynamics_sincos(
    np.pi/4, 3, 3
)

X = time_series_generator(
    dynamics, 
    np.random.normal(0, 1, 6),
    500,
    0.2
)

dataset = TimeSeriesData(
    X[:-1, :],
    X[1:, :],
    lag=1,
    train_val_test_split=[0.5, 0.3, 0.2]
)

X_train, y_train = dataset.train_data()
X_val, y_val = dataset.val_data()
X_test, y_test = dataset.test_data()

In [ ]:
class RascuttiModel:
    def __init__(
        self,
        kernel:str|list="gaussian",
        lam:float=1e-9,
        rho:float=1e-9,
        kernel_perterbation = 1e-9,
        **kwargs
    ):
        # Build kernel functions
        # Case 1: One input
        # Case 2: List to be applied across all target dimensions
        # Case 3: One Kernel per input and target dimension. Each upper list represents an output dimension, and each inner is an input dimension
        if kernel == "gaussian" or kernel == "rbf":
            if "bandwidth" not in kwargs:
                raise Exception("Please specify rbf bandwidths")
            
            if type(kwargs["bandwidth"]) == list:
                if type(kwargs["bandwidth"][0]) == list:
                    kernels = []
                    for i in kwargs["bandwidth"]:
                        self._check(len(i) == len(kwargs["bandwidth"][0]), msg="All output dimensions have the same number of input dimensions, and require the same number of kernels")
                        k_row = []
                        for b in i:
                            k_row.append(GaussianKernel(bandwidth=b))
                        kernels.append(k_row)
                else:
                    kernels = [GaussianKernel(bandwidth=b) for b in kwargs["bandwidth"]]

            else:
                kernels = [GaussianKernel(bandwidth=kwargs["bandwidth"])]
        else:
            raise NotImplementedError("Only RBF kernel is implemented")
            
        self.lam = lam
        self.rho = rho
        self.kernel_perterbation = kernel_perterbation
        self.kernel_shapes = [np.inf, np.inf] # Input dim, Target dim

        self.kernel_factory = self._construct_kernel_factory(kernels)

    def _construct_kernel_factory(self, kernels):
        # Determine shape of kernels
        # Case 1: One input
        if type(kernels[0]) != list and len(kernels) == 1:
            def kernel_factory(input_dimension, target_dimension):
                return kernels[0]

            return kernel_factory
            
        # Case 2: One set of kernels for all target dimensions
        elif type(kernels[0]) != list and len(kernels) > 1:
            self.kernel_shapes[0] = len(kernels)
            def kernel_factory(input_dimension, target_dimension):
                self._check(input_dimension < len(kernels), msg=f"Input dimension index is too high: index = {input_dimension}")
                return kernels[input_dimension]
            
            return kernel_factory
        
        # Case 3: One kernel for each input and target dimension
        else:
            self.kernel_shapes = [len(kernels[0]), len(kernels)]
            def kernel_factory(input_dimension, target_dimension):
                self._check(target_dimension < len(kernels), msg=f"Target dimension index is too high: index = {target_dimension}")
                self._check(input_dimension < len(kernels[target_dimension]), msg=f"Input dimension index is too high: index = {target_dimension}")
                return kernels[target_dimension][input_dimension]
            return kernel_factory

    def _get_problem_dimensions(self, arr):
        if len(arr.shape) == 1:
            N = arr.shape[0]
            d = 1
        else:
            N = arr.shape[0]
            d = arr.shape[-1]
        return N, d
    
    def _check(self, statement, msg="Assertion Error"):
        if not statement:
            raise Exception(msg)
        
    def fit(self, X:np.array, y:np.array):
        # Get problem dimensions
        N, Xd = self._get_problem_dimensions(X)
        N_check, Yd = self._get_problem_dimensions(y)

        self._check(N == N_check, "Both X and y must have the same number of data points")

        # Ensure correct number of Kernels
        self._check(Xd <= self.kernel_shapes[0], msg=f"Not enough Kernels for input dimension of {Xd} (N kernels = {self.kernel_shapes[0]})")
        self._check(Yd <= self.kernel_shapes[1], msg=f"Not enough Kernels for output dimension of {Yd} (N kernels = {self.kernel_shapes[1]})")

        # Constuct SOCP constaints.
        t = cp.Variable()
        # alphas = cp.Variable(shape=(N, Xd, Yd))
        alphas = [[cp.Variable(N)]*Yd]*Xd
        # u = cp.Variable(shape=(Xd, Yd))
        u = [[cp.Variable()]*Yd]*Xd
        # v = cp.Variable(shape=(Xd, Yd))
        v = [[cp.Variable()]*Yd]*Xd


        socp_constraints = []
        # Case 1: Same kernels for each output dimension
        if self.kernel_shapes[1] == np.inf:
            summed_predictors = 0

            for j in range(Xd):
                kernel_matrix = self.kernel_factory(j, -1)(X[..., j], X[..., j])
                sqrt_kernel = np.linalg.cholesky(
                    kernel_matrix + self.kernel_perterbation*np.diag(np.random.random(kernel_matrix.shape[0]))
                )

                for l in range(Yd):
                    socp_constraints.append(
                        cp.SOC(1, cp.hstack([0.5, sqrt_kernel@alphas[j][l]]))
                    )

                    socp_constraints.append(
                        cp.SOC(v[j][l], kernel_matrix@alphas[j][l])
                    )
                    
                    socp_constraints.append(
                        cp.SOC(u[j][l], sqrt_kernel@alphas[j][l])
                    )

                summed_predictors += sp.linalg.block_diag(*[kernel_matrix]*Yd)@ cp.hstack(alphas[j])
            
            summed_predictors = y.T.reshape(-1, 1).squeeze() - summed_predictors

            socp_constraints.append(
                cp.SOC(t, cp.hstack([0.5, summed_predictors]))
            )

        # Case 2: Different kernels for each output dimension
        else:
            raise NotImplementedError("Multiple not implemented")
        
        v_sum = cp.sum([j for i in v for j in i])
        u_sum = cp.sum([j for i in u for j in i])
        
        prob = cp.Problem(
            cp.Minimize(1/(2*N)*t + (self.lam/np.sqrt(N))*v_sum + self.rho*u_sum),
            socp_constraints
        )

        prob.solve()

        self.alphas = alphas

    def predict(self, X):
        pass


In [8]:
t1 = time()
model = RascuttiModel(
    kernel="rbf",
    bandwidth = 1
)
t2 = time()
model.fit(X_train, y_train)
t3 = time()

print(
f"""
Setup Time: {t2-t1}
Train Time: {t3-t2}
Total Time: {t3-t1}
"""
)


Setup Time: 0.00017762184143066406
Train Time: 33.82860016822815
Total Time: 33.82877779006958



In [9]:
t1 = time()
model = RascuttiOld(
    kernel="rbf",
    bandwidth = 1
)
t2 = time()
model.fit(X_train, y_train)
t3 = time()

print(
f"""
Setup Time: {t2-t1}
Train Time: {t3-t2}
Total Time: {t3-t1}
"""
)


Setup Time: 6.937980651855469e-05
Train Time: 24.59089183807373
Total Time: 24.59096121788025

